# 🫁 Chest X-Ray AI: Clinical Pneumonia Triage
### Deep Fine-Tuning DenseNet121 at 448px (Experiment J)

This notebook represents the final iteration of the model training process. We utilize a **DenseNet121** architecture with selective unfreezing (Deep Fine-Tuning), high-resolution processing (448px), and heavy data augmentation to maximize the recall of Pneumonia detection while minimizing False Positives. We also incorporate **MLOps Checkpointing** and **Test-Time Augmentation (TTA)** for optimal performance.

In [ ]:
import os
import shutil
import random

# =====================================================================
# 1. ENVIRONMENT SETUP & AUTHENTICATION
# =====================================================================
# Retrieve credentials securely from Colab's Secret Manager (Optional)
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except ImportError:
    pass

print("📥 Downloading dataset via Kaggle API...")
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia > /dev/null
!unzip -q chest-xray-pneumonia.zip -d dataset/
print("✅ Dataset downloaded and extracted.")

# =====================================================================
# 2. DATASET RESTRUCTURING (FIXING ORIGINAL SPLIT)
# =====================================================================
# The original dataset has a flawed validation split (only 16 images).
# We merge train/val and recreate a robust 80/20 split.
base_dir = './dataset/chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
classes = ['NORMAL', 'PNEUMONIA']

print("🔄 Restructuring data for an robust train/validation split...")

for cls in classes:
    train_imgs = os.listdir(os.path.join(train_dir, cls))
    val_imgs = os.listdir(os.path.join(val_dir, cls))
    all_imgs = train_imgs + val_imgs
    random.shuffle(all_imgs)

    split_idx = int(len(all_imgs) * 0.8)
    new_train, new_val = all_imgs[:split_idx], all_imgs[split_idx:]

    for img in val_imgs:
        shutil.move(os.path.join(val_dir, cls, img), os.path.join(train_dir, cls, img))
    for img in new_val:
        shutil.move(os.path.join(train_dir, cls, img), os.path.join(val_dir, cls, img))

print("✅ Data splits fixed.")

### ⚙️ Dataloaders & Heavy Augmentation
We scale the inputs to `448x448` and apply heavy augmentation to simulate distinct domain shifts (scanner differences, exposure) forcing the model to ignore color casts.

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. HEAVY AUGMENTATION AND 448x448 RESOLUTION
transform_train = transforms.Compose([
    transforms.Resize((448, 448)), # 🌟 Double resolution!
    transforms.RandomRotation(15), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1), # Simulate different machines
    transforms.RandomGrayscale(p=0.1), # Force model to ignore color tints
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((448, 448)), # Validation is only resized, no distortion
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. LOAD DATASETS (Batch size 16 to prevent memory overload at 448px)
batch_size = 16
train_dataset = datasets.ImageFolder(train_dir, transform=transform_train)
val_dataset = datasets.ImageFolder(val_dir, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"✅ DataLoaders successfully created.")
print(f"Training batches: {len(train_loader)} (of {batch_size} images each)")
print(f"Validation batches: {len(val_loader)}")

### 🧠 Architecture Strategy: Selective Unfreezing
We freeze the early layers of DenseNet121, but unfreeze the deep dense blocks. This enables Deep Fine-Tuning, allowing the most complex, abstract feature filters to adapt entirely to the pulmonary parenchyma structures.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Initializing on {device}")

# 1. LOAD ARCHITECTURE
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

# 2. SELECTIVE UNFREEZING
# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the last dense blocks to specialize in lungs
blocks_to_train = ['denseblock3', 'transition3', 'denseblock4', 'norm5']
for name, param in model.named_parameters():
    if any(block in name for block in blocks_to_train):
        param.requires_grad = True

# Reset the final classifier (this will have requires_grad=True by default)
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 2)
model = model.to(device)

# 3. OPTIMIZER WITH LOW LEARNING RATE (Micro-surgery)
# Instruct Adam to only update unfrozen parameters
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

# Using neutral CrossEntropyLoss because the architecture is already highly sensitive
criterion = nn.CrossEntropyLoss()

print("✅ DenseNet121 configured for 448px.")
print("✅ Blocks 3 and 4 unfrozen.")
print("✅ Ready for training!")

### 🏃‍♂️ Training Loop with MLOps Checkpointing
We train for 10 epochs. Deep fine-tuning can lead to memory amnesia or overfitting (as recorded in Experiment J Logs). We implement a checkpointing system to physically save the "brain state" of the model at its highest validation success.

In [ ]:
import time
import copy

num_epochs = 10
best_acc = 0.0
best_loss = float('inf')
# Temporarily store the "brain" when it improves
best_model_wts = copy.deepcopy(model.state_dict())

print(f"🏃‍♂️ Starting training of {num_epochs} epochs with Checkpointing...")
start_time = time.time()

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 20)

    # Each epoch has a training and validation phase
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Model in training mode (applies Dropout and Batchnorm)
            dataloader = train_loader
        else:
            model.eval()   # Model in evaluation mode (turns off Dropout)
            dataloader = val_loader

        running_loss = 0.0
        running_corrects = 0
        total_samples = 0

        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = running_corrects.double() / total_samples

        print(f'👉 {phase.capitalize():<5} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

        # 🌟 THE MAGIC OF CHECKPOINTING 🌟
        if phase == 'val' and epoch_loss < best_loss:
            print(f"   🏆 NEW RECORD! Val Loss dropped from {best_loss:.4f} to {epoch_loss:.4f}.")
            print("   💾 Saving this version of the model as the best...")
            best_loss = epoch_loss
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), 'densenet_finetuned_best.pth')

time_elapsed = time.time() - start_time
print('\n' + '='*50)
print(f'✅ Training completed in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
print(f'🌟 Best Val Loss: {best_loss:.4f} (Acc: {best_acc:.4f})')

# Load best model weights into memory
model.load_state_dict(best_model_wts)
print("🧠 The model in memory now holds the weights of its best epoch.")
print('='*50)

### 🩺 Pure Inference on External Test Set (Domain Shift)
We now evaluate the model against the completely unseen test. Due to clinical asymmetry, we are most interested in the False Negatives count.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print("🚀 Starting Final Evaluation (Test Set)...")
model.eval()

test_dir = os.path.join(base_dir, 'test')

transform_test = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

ds_test = datasets.ImageFolder(test_dir, transform=transform_test)
dl_test = DataLoader(ds_test, batch_size=16, shuffle=False)

print(f"🔍 Evaluating {len(ds_test)} giant radiographs. Fingers crossed!...")
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in dl_test:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("\n" + "=" * 50)
print("🏆 TEST SET RESULTS: PURE INFERENCE (448px)")
print("=" * 50)
print(f"Overall Accuracy: {accuracy:.4f}")
print(f"   🚨 False Negatives (Undetected sick): {fn}")
print(f"   💸 False Positives (Healthy flagged): {fp}")
print(f"   ✅ Hits: {tp} Pneumonia, {tn} Normal")
print("-" * 50)
print(classification_report(all_labels, all_preds, target_names=['Normal', 'Pneumonia']))
print("=" * 50)

### 🩺 The Medical Committee: 5-Way Test-Time Augmentation (TTA)
We evaluate the same patients through 5 distinct simulated environments (Rotations, Zoom, Contrast Shifts) and use a hard voting consensus.

In [ ]:
print("🚀 Starting THE MEDICAL COMMITTEE: 5-Way TTA (448px)...")
model.eval()

# 1. THE 5 PERSPECTIVES
t_orig = transform_test
t_rot_r = transforms.Compose([
    transforms.Resize((448, 448)), transforms.RandomRotation((15, 15)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
t_rot_l = transforms.Compose([
    transforms.Resize((448, 448)), transforms.RandomRotation((-15, -15)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
t_zoom = transforms.Compose([
    transforms.Resize((512, 512)), transforms.CenterCrop(448), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
t_contrast = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dl_o = DataLoader(datasets.ImageFolder(test_dir, transform=t_orig), batch_size=16, shuffle=False)
dl_rr = DataLoader(datasets.ImageFolder(test_dir, transform=t_rot_r), batch_size=16, shuffle=False)
dl_rl = DataLoader(datasets.ImageFolder(test_dir, transform=t_rot_l), batch_size=16, shuffle=False)
dl_z = DataLoader(datasets.ImageFolder(test_dir, transform=t_zoom), batch_size=16, shuffle=False)
dl_c = DataLoader(datasets.ImageFolder(test_dir, transform=t_contrast), batch_size=16, shuffle=False)

print(f"🔍 The Committee is reviewing cases. Evaluating...")
all_labels = []
tta_preds = []

with torch.no_grad():
    for (b_o, lbls), (b_rr, _), (b_rl, _), (b_z, _), (b_c, _) in zip(dl_o, dl_rr, dl_rl, dl_z, dl_c):
        lbls = lbls.to(device)

        _, p_o = torch.max(model(b_o.to(device)), 1)
        _, p_rr = torch.max(model(b_rr.to(device)), 1)
        _, p_rl = torch.max(model(b_rl.to(device)), 1)
        _, p_z = torch.max(model(b_z.to(device)), 1)
        _, p_c = torch.max(model(b_c.to(device)), 1)

        sum_votes = p_o + p_rr + p_rl + p_z + p_c

        # 🌟 RULE: If 3 or more "doctors" say pneumonia, it is pneumonia
        final_preds = (sum_votes >= 3).int()

        tta_preds.extend(final_preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

cm = confusion_matrix(all_labels, tta_preds)
tn, fp, fn, tp = cm.ravel()
acc = (tp + tn) / (tp + tn + fp + fn)

print("\n" + "=" * 50)
print("🏆 SUPREME RESULTS: 5-WAY TTA (Medical Committee)")
print("=" * 50)
print(f"Overall Accuracy: {acc:.4f}")
print(f"   🚨 False Negatives: {fn}")
print(f"   💸 False Positives: {fp}")
print(f"   ✅ Hits: {tp} Pneumonia, {tn} Normal")
print("-" * 50)
print(classification_report(all_labels, tta_preds, target_names=['Normal', 'Pneumonia']))